In [1]:
!pip install transformers torch

In [ ]:
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from google.colab import drive
drive.mount('/content/drive')

INPUT_PATH = "/content/drive/MyDrive/FTD/DL/cleaned_articles_english_v2.csv"
OUTPUT_ARTICLES = "/content/drive/MyDrive/FTD/DL/articles_with_climatebert.csv"
OUTPUT_DAILY = "/content/drive/MyDrive/FTD/DL/climatebert_daily_signal.csv"

Mounted at /content/drive


In [ ]:
df = pd.read_csv(INPUT_PATH)
print(f"Loaded {len(df)} articles")
print(f"Date range: {df['news_date'].min()} → {df['news_date'].max()}")

# Clean: ensure text exists and is reasonable length
df = df[df['text_clean'].notna() & (df['text_clean'].str.len() > 100)].copy()
print(f"After filtering: {len(df)} articles with usable text")

Loaded 8866 articles
Date range: 2015-02-18 → 2025-12-31
After filtering: 8866 articles with usable text


In [ ]:
MODEL_NAME = "climatebert/distilroberta-base-climate-detector"

print("Loading ClimateBERT model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()
print(f"Model loaded on {device}")

# Check label mapping
# ClimateBERT climate-detector outputs: 0 = not climate, 1 = climate
print(f"Labels: {model.config.id2label}")

Loading ClimateBERT model...


config.json:   0%|          | 0.00/887 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/329M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: climatebert/distilroberta-base-climate-detector
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded on cuda
Labels: {0: 'no', 1: 'yes'}


In [ ]:
BATCH_SIZE = 32
MAX_LENGTH = 512  # ClimateBERT's max context

all_probs = []
all_labels = []

texts = df['text_clean'].tolist()
n_batches = (len(texts) + BATCH_SIZE - 1) // BATCH_SIZE

print(f"\nProcessing {len(texts)} articles in {n_batches} batches...")

for i in range(0, len(texts), BATCH_SIZE):
    batch_texts = texts[i : i + BATCH_SIZE]

    # Truncate very long texts to first 512 tokens
    # ClimateBERT was fine-tuned on shorter passages
    # Using only the beginning is standard practice
    inputs = tokenizer(
        batch_texts,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        # Softmax to get probabilities
        probs = torch.softmax(outputs.logits, dim=-1)
        # Column 1 = probability of "climate-related"
        climate_probs = probs[:, 1].cpu().numpy()
        # Binary prediction at 0.5 threshold
        labels = (climate_probs >= 0.5).astype(int)

    all_probs.extend(climate_probs.tolist())
    all_labels.extend(labels.tolist())

    if (i // BATCH_SIZE + 1) % 50 == 0 or i + BATCH_SIZE >= len(texts):
        pct = min(i + BATCH_SIZE, len(texts)) / len(texts) * 100
        avg_prob = np.mean(all_probs)
        climate_rate = np.mean(all_labels)
        print(f"  {pct:.0f}% done | avg_prob={avg_prob:.3f} | climate_rate={climate_rate:.2%}")

# Add to dataframe
df['climatebert_prob'] = all_probs
df['climatebert_label'] = all_labels

print(f"\nClimateBERT results:")
print(f"  Climate articles: {df['climatebert_label'].sum()} / {len(df)} ({df['climatebert_label'].mean():.2%})")
print(f"  Mean probability: {df['climatebert_prob'].mean():.3f}")


Processing 8866 articles in 278 batches...
  18% done | avg_prob=0.575 | climate_rate=59.19%
  36% done | avg_prob=0.618 | climate_rate=63.31%
  54% done | avg_prob=0.639 | climate_rate=65.15%
  72% done | avg_prob=0.641 | climate_rate=65.41%
  90% done | avg_prob=0.647 | climate_rate=66.04%
  100% done | avg_prob=0.652 | climate_rate=66.77%

ClimateBERT results:
  Climate articles: 5920 / 8866 (66.77%)
  Mean probability: 0.652


In [ ]:
daily = df.groupby('news_date').agg(
    # Volume
    n_articles=('climatebert_prob', 'size'),

    # ClimateBERT climate signal
    n_climate_bert=('climatebert_label', 'sum'),
    climate_prob_mean=('climatebert_prob', 'mean'),
    climate_prob_max=('climatebert_prob', 'max'),
    climate_prob_std=('climatebert_prob', 'std'),

    # Existing GDELT topic flags (already in the data)
    frac_core_climate=('is_core_climate', 'mean'),
    frac_policy=('is_policy_regulation', 'mean'),
    frac_physical=('is_physical_risk', 'mean'),
    frac_energy=('is_energy_geopolitics', 'mean'),
    frac_cleantech=('is_cleantech', 'mean'),
    frac_activism=('is_activism_litigation', 'mean'),
).reset_index()

# Derived features
daily['climate_share_bert'] = daily['n_climate_bert'] / daily['n_articles']
daily['log_n_articles'] = np.log1p(daily['n_articles'])

# Fill std with 0 for days with 1 article
daily['climate_prob_std'] = daily['climate_prob_std'].fillna(0)

print(f"\nDaily signal: {len(daily)} days")
print(f"  Articles/day: mean={daily['n_articles'].mean():.1f}, median={daily['n_articles'].median():.0f}")
print(f"  Climate share (BERT): {daily['climate_share_bert'].mean():.2%}")
print(daily.head(10).to_string())


Daily signal: 3092 days
  Articles/day: mean=2.9, median=2
  Climate share (BERT): 66.71%
    news_date  n_articles  n_climate_bert  climate_prob_mean  climate_prob_max  climate_prob_std  frac_core_climate  frac_policy  frac_physical  frac_energy  frac_cleantech  frac_activism  climate_share_bert  log_n_articles
0  2015-02-18           4               2           0.537681          0.997828          0.447359               1.00          0.0           0.00         0.50             0.0            0.0                0.50        1.609438
1  2015-02-19           3               3           0.933869          0.998231          0.110687               1.00          0.0           0.00         0.00             0.0            0.0                1.00        1.386294
2  2015-02-20           2               2           0.969741          0.997871          0.039781               1.00          0.0           0.00         0.50             0.0            0.0                1.00        1.098612
3  2015-02-21

In [ ]:
#validation
# Compare ClimateBERT labels with rule-based GDELT flags

print(f"\n=== VALIDATION ===")
print("ClimateBERT vs GDELT is_core_climate flag:")
if 'is_core_climate' in df.columns:
    ct = pd.crosstab(df['climatebert_label'], df['is_core_climate'],
                     rownames=['BERT'], colnames=['GDELT'])
    print(ct)
    agreement = ((df['climatebert_label'] == 1) & (df['is_core_climate'] == 1)).sum()
    total_either = ((df['climatebert_label'] == 1) | (df['is_core_climate'] == 1)).sum()
    print(f"Agreement rate: {agreement/total_either:.2%}")

# Show high-confidence climate articles
print(f"\nTop climate articles (highest BERT probability):")
top = df.nlargest(5, 'climatebert_prob')
for _, row in top.iterrows():
    print(f"  [{row['news_date']}] prob={row['climatebert_prob']:.3f} | {str(row['text_clean'])[:120]}...")


=== VALIDATION ===
ClimateBERT vs GDELT is_core_climate flag:
GDELT     0     1
BERT             
0      1744  1202
1      1993  3927
Agreement rate: 55.14%

Top climate articles (highest BERT probability):
  [2025-04-03] prob=0.999 | DEWA wins 4 Best Business Awards 2025 in the UK Dubai Electricity and Water Authority (DEWA) has won four awards at the ...
  [2024-04-22] prob=0.999 | Environment awards spotlight green initiatives The awards mark World Environment Day and recognise projects within 12 ca...
  [2024-10-21] prob=0.999 | The World Sustainable Hospitality Alliance (the Alliance) proudly announces its partnership with the International Platf...
  [2024-07-04] prob=0.999 | In today’s competitive business landscape, companies are increasingly recognising the importance of adopting effective e...
  [2016-03-29] prob=0.999 | The species conservation award recognises individuals and organisations making extraordinary efforts to protect a single...


In [ ]:
df.to_csv(OUTPUT_ARTICLES, index=False)
daily.to_csv(OUTPUT_DAILY, index=False)